# 02. Thực nghiệm tiền xử lý trên crop có nhãn
 Dùng API hiện tại của `PlatePreprocessor`, `PlateOCREngine`, `PlatePostprocessor`.
 Đo cùng một tập crop cho hai đầu vào: ảnh gốc và CLAHE + bilateral.

 Đây là thí nghiệm **OCR trên bbox ground truth**: vị trí biển đã được cung cấp. Không dùng kết quả để kết luận detector hoặc web nhận diện được ảnh toàn cảnh.
 Engine YOLO hiện tại có các nhánh thử lại riêng; thí nghiệm đo tác động của đầu vào tăng cường lên engine đó, không cô lập từng thuật toán bên trong.
 Mặc định chọn cố định 2 ảnh đầu mỗi nhóm (không chọn theo kết quả OCR). Có thể đặt `PER_CATEGORY = None` để chạy toàn bộ.

In [ ]:
from pathlib import Path
import sys
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/core').is_dir() and (p / 'notebooks').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from notebooks._support import read_json, read_image, checked_crop, sha256
%matplotlib inline
plt.rcParams.update({'figure.figsize': (11, 4), 'font.size': 10})
print('Project:', ROOT)
print('Python:', sys.executable)

In [ ]:
from notebooks._support import load_synthetic_samples
from src.core.preprocessor import PlatePreprocessor
from src.core.postprocessor import PlatePostprocessor
from src.evaluation.metrics import normalize_text, levenshtein_distance, timing_stats

PER_CATEGORY = 2
samples = load_synthetic_samples()
chosen = []
for category in sorted({s['category'] for s in samples}):
    group = [s for s in samples if s['category'] == category]
    chosen.extend(group if PER_CATEGORY is None else group[:PER_CATEGORY])
crops = []
for sample in chosen:
    image = read_image(sample['path'])
    for index, target in enumerate(sample['truth']):
        crops.append({'id': f"{sample['file']}#{index}", 'category': sample['category'],
                      'expected': target['text'], 'image_sha256': sha256(sample['path']),
                      'image': checked_crop(image, target['bbox'])})
if not crops:
    raise ValueError('Không có crop có nhãn để thử nghiệm.')
display(pd.DataFrame([{k:v for k,v in item.items() if k != 'image'} for item in crops]))
print('Số crop được chọn / tổng bbox:', len(crops), '/', sum(len(s['truth']) for s in samples))

In [ ]:
sample_crop = crops[0]['image']
enhanced = PlatePreprocessor.enhance_for_ocr(sample_crop)
top, bottom = PlatePreprocessor.split_two_line_plate(sample_crop)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, image, title in zip(axes, [sample_crop, enhanced, top, bottom],
                           ['Gốc', 'CLAHE + bilateral', 'Cắt phần trên', 'Cắt phần dưới']):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if image.ndim == 3 else image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Ảnh minh họa:', crops[0]['id'])
print('Cắt tại tỷ lệ 0.48 là heuristic của module; cần kiểm tra có cắt qua ký tự hay không.')

## Chạy OCR cục bộ và đo kết quả
 Chỉ dùng `LP_ocr.pt`, CPU, 2 luồng Torch và `recognize_local_yolo`; không gọi 9Router, API web hoặc database.
 Một lượt warmup cho mỗi biến thể. Thời gian từng mẫu gồm tiền xử lý tương ứng, OCR và định dạng; không gồm giải mã ảnh hoặc nạp model.
 Nhãn chỉ dùng để chấm điểm sau dự đoán. CER so chuỗi đã bỏ dấu phân cách; kiểm tra định dạng không chứng minh đúng ký tự.

In [ ]:
import platform
import torch
from time import perf_counter
from src.core.ocr_engine import PlateOCREngine

weights = ROOT / 'models/LP_ocr.pt'
if not weights.is_file():
    raise FileNotFoundError('Thiếu models/LP_ocr.pt; notebook không tự tải model.')
torch.set_num_threads(2)
engine = PlateOCREngine(engine_type='yolo')
if engine.yolo_ocr is None:
    raise RuntimeError('Không nạp được YOLO OCR; dừng thay vì ghi điểm giả.')
engine.yolo_ocr.to('cpu')
assert engine.ninerouter_client is None
print({'python': platform.python_version(), 'torch': torch.__version__, 'opencv': cv2.__version__,
       'device': 'cpu', 'threads': torch.get_num_threads(), 'weights_sha256': sha256(weights)})
for name in ['ocr_engine.py', 'preprocessor.py', 'postprocessor.py']:
    print(name, sha256(ROOT / 'src/core' / name))
# The OCR entry point expects three-channel BGR. Replicate grayscale into BGR.
variants = {'original': lambda im: im.copy(),
            'clahe_bilateral': lambda im: cv2.cvtColor(
                PlatePreprocessor.enhance_for_ocr(im), cv2.COLOR_GRAY2BGR)}
for transform in variants.values():
    engine.recognize_local_yolo(transform(crops[0]['image']))
measurements = []
for i, item in enumerate(crops):
    # Alternate order to reduce consistent first-variant timing bias.
    order = list(variants.items()) if i % 2 == 0 else list(reversed(list(variants.items())))
    for variant, transform in order:
        started = perf_counter()
        result = engine.recognize_local_yolo(transform(item['image']))
        text = PlatePostprocessor.format_vietnamese_plate(
            result.get('raw_text', ''), plate_type=result.get('plate_type'), text_lines=result.get('text_lines'))
        valid = PlatePostprocessor.validate_plate(text)
        prediction = text if valid else ''
        elapsed = (perf_counter() - started) * 1000
        expected, actual = normalize_text(item['expected']), normalize_text(prediction)
        measurements.append({'id': item['id'], 'category': item['category'], 'variant': variant,
                             'expected': item['expected'], 'raw': result.get('raw_text', ''),
                             'actual': prediction, 'valid_format': valid,
                             'exact': expected == actual, 'edit_errors': levenshtein_distance(expected, actual),
                             'gt_chars': len(expected), 'latency_ms': elapsed})
measured = pd.DataFrame(measurements).sort_values(['id', 'variant'])
display(measured[['id', 'variant', 'expected', 'actual', 'exact', 'latency_ms']])

In [ ]:
summaries = []
for variant, group in measured.groupby('variant'):
    summaries.append({'variant': variant, 'crops': len(group), 'exact': int(group.exact.sum()),
                      'exact_rate': group.exact.mean(), 'cer': group.edit_errors.sum() / group.gt_chars.sum(),
                      **timing_stats(group.latency_ms.tolist())})
summary = pd.DataFrame(summaries).set_index('variant')
display(summary)
paired = measured.pivot(index='id', columns='variant', values='exact')
gained = (~paired['original'] & paired['clahe_bilateral']).sum()
lost = (paired['original'] & ~paired['clahe_bilateral']).sum()
print(f'Trên đúng {len(paired)} crop: tăng cường sửa đúng thêm {gained}, làm sai thêm {lost}.')
print('Kết luận chỉ áp dụng cho tập crop này. Không suy FPS web từ thời gian OCR crop.')

## Hậu xử lý hiện tại: kiểm tra cấu trúc và tách dòng
 Các chuỗi dưới đây là ví dụ kiểm tra hàm, **không phải dự đoán từ ảnh** và không đưa vào tỷ lệ OCR.
 Đã bỏ các lời gọi cũ không tồn tại (`fix_optical_confusions`, `is_valid_vietnamese_plate`).

In [ ]:
examples = [
    ('51A13883', None, 'one_line'),
    ('29MD100160', ['29MD1', '00160'], 'two_line'),
    ('54H45555', ['54H4', '5555'], 'two_line'),
    ('PARKING2026', None, None),
    ('', None, None),
]
rows = []
for raw, lines, plate_type in examples:
    formatted = PlatePostprocessor.format_vietnamese_plate(raw, plate_type=plate_type, text_lines=lines)
    rows.append({'input': raw, 'text_lines': lines, 'formatted': formatted,
                 'valid_format': PlatePostprocessor.validate_plate(formatted)})
display(pd.DataFrame(rows))
print('Regex chỉ kiểm tra cấu trúc được phần mềm hỗ trợ; chuỗi hợp lệ vẫn có thể đọc sai biển thực tế.')